# Import packages

In [ ]:
import numpy as np
import time
import sys
import traceback


# Parameters

In [ ]:
total_steps=2500000
save_steps=100
epsilon=0.001
relaxation_steps=10
density_threshold=1e30
total_time=0.0
a=2.257
b=1.385
c=0.753
sigma=0.5
mass_norm=0.1
scale_norm=0.1
tag='2026xxxx'
input_dir='C:/Users/user/Desktop/.venv/gravothermal-main/result/initial/' + tag + '/'
output_file='C:/Users/user/Desktop/.venv/gravothermal-main/result/'+tag+'.txt'

# Programs

In [ ]:
def display():
  print('Initial profile:',tag)
  print('Initial time:',total_time)
  print('Total step:',total_steps)
  print('Save step:',save_steps)
  print('Abs(delta u/u):',epsilon)
  print('Cross section (sigma):',sigma)
  print('Conduction parameter (a,b,c):',a,',',b,',',c)
  print('Baryon parameter (mass_norm,scale):',mass_norm,',',scale_norm)


In [ ]:
def MbaryonP(myr,mymass,myscale):
  return mymass*(1.+myscale**2/myr**2)**-1.5
def MbaryonH(myr,mymass,myscale):
  return mymass*(1.+myscale/myr)**-2.
def MbaryonSPL(myr,mymass):
  return mymass*myr**.6


In [ ]:
class SimulationState:
  def initialize(self):
    try:
      self.nameR=input_dir+'RList-'+str(tag)+'.txt'
      self.nameRho=input_dir+'RhoList-'+str(tag)+'.txt'
      self.nameM=input_dir+'MList-'+str(tag)+'.txt'
      self.nameu=input_dir+'uList-'+str(tag)+'.txt'
      self.nameL=input_dir+'LList-'+str(tag)+'.txt'

      self.RList=np.loadtxt(self.nameR)
      self.RhoList=np.loadtxt(self.nameRho)
      self.MList=np.loadtxt(self.nameM)
      self.uList=np.loadtxt(self.nameu)
      self.LList=np.loadtxt(self.nameL)

      self.NoLayers=len(self.RList)

      self.MhyList=np.zeros(self.NoLayers)
      self.vList=np.zeros(self.NoLayers)
      self.pList=np.zeros(self.NoLayers)
      self.aList=np.zeros(self.NoLayers)

      self.vList=np.sqrt(2./3.)*np.sqrt(self.uList)

      i = np.arange(self.NoLayers-1)
      self.MhyList[i]=self.MList[i]+MbaryonP(self.RList[i],mass_norm,scale_norm)

      print('Number of layers:',self.NoLayers)
      print('Initial r inner most:',self.RList[0])
      print('Initial r next to outer most:',self.RList[-2])
      print('Initial rho inner most:',self.RhoList[0])
      print('Initial rho next to outer most:',self.RhoList[-2])

    except Exception as e:
      print('Failed to initialize simulation state:',e)

  def checkForAbnormalState(self):
    if self.RhoList[0]>density_threshold:
      print('Rho reaches threshold!')
      return True

    if self.RList[0]<0:
      print('R[0] is negative!')
      return True

    if np.isnan(self.RList[0]):
      print('R is nan!')
      return True

    return False

In [ ]:
from numba import njit, prange

# --- Numba-Accelerated Heavy Math Kernels ---

@njit(fastmath=True)
def numba_conduction_step(LList, MList, uList, RhoList, pList, aList, epsilon):
    NoLayers = len(LList)
    deltaUcoeff = np.zeros(NoLayers)
    
    deltaUcoeff[0] = -(LList[0] / MList[0] / uList[0])
    for i in range(1, NoLayers - 1):
        deltaUcoeff[i] = -((LList[i] - LList[i-1]) / (MList[i] - MList[i-1])) / uList[i]

    deltat = epsilon / np.max(np.abs(deltaUcoeff))

    deltaU = deltaUcoeff * uList * deltat
    uList += deltaU

    pList[:] = (2.0 / 3.0) * (RhoList * uList)
    aList[:] = (2.0 / 3.0) * (RhoList ** (-2.0 / 3.0)) * uList

    return deltat


@njit(fastmath=True)
def numba_setup_matrix(Hydromat, Hydrob, RList, pList, MhyList, RhoList, NoLayers):
    # Clear matrix and vector quickly
    Hydromat.fill(0.0)
    Hydrob.fill(0.0)

    R0 = RList[0]
    R0_2 = R0 * R0
    R0_3 = R0_2 * R0
    R0_4 = R0_2 * R0_2
    R1 = RList[1]
    R1_2 = R1 * R1
    R1_R0 = R1 - R0
    R1_R0_sum_of_squares = R1_2 + R1 * R0 + R0_2
    R1_3_minus_R0_3_inv = 1.0 / (R1_R0 * R1_R0_sum_of_squares)

    Hydromat[0, 0] = (
        8.0 * R0 * (pList[1] - pList[0]) +
        20.0 * R0_4 * (pList[0] / R0_3 + pList[1] * R1_3_minus_R0_3_inv) +
        3.0 * MhyList[0] * R1 * R0_2 * (-RhoList[0] / R0_3 + RhoList[1] * R1_3_minus_R0_3_inv)
    )

    Hydromat[0, 1] = (
        MhyList[0] * (RhoList[0] + RhoList[1]) -
        20.0 * R0_2 * pList[1] * R1_2 * R1_3_minus_R0_3_inv -
        3.0 * MhyList[0] * RhoList[1] * R1_3_minus_R0_3_inv * (R1**3 if False else R1**3) # Safe representation
    )

    Hydrob[0] = (
        -4.0 * R0_2 * (pList[1] - pList[0]) -
        MhyList[0] * (RhoList[0] + RhoList[1]) * R1
    )

    for i in range(1, NoLayers - 2):
        Ri = RList[i]
        Ri_2 = Ri * Ri
        Ri_4 = Ri_2 * Ri_2
        Ri_1 = RList[i-1]
        Ri_1_2 = Ri_1 * Ri_1
        Ri1 = RList[i+1]
        Ri1_2 = Ri1 * Ri1
        
        Ri_Ri_1 = Ri - Ri_1
        Ri_Ri_1_sum_of_squares = Ri_2 + Ri * Ri_1 + Ri_1_2
        Ri_3_minus_Ri_1_3_inv = 1.0 / (Ri_Ri_1 * Ri_Ri_1_sum_of_squares)

        Ri1_Ri = Ri1 - Ri
        Ri1_Ri_sum_of_squares = Ri1_2 + Ri1 * Ri + Ri_2 # Note: Ri_2 here corresponds to Ri^2
        Ri1_3_minus_Ri_3_inv = 1.0 / (Ri1_Ri * Ri1_Ri_sum_of_squares)

        Hydromat[i, i-1] = (
            -MhyList[i] * (RhoList[i] + RhoList[i+1])
            - 20.0 * Ri_2 * pList[i] * Ri_1_2 * Ri_3_minus_Ri_1_3_inv
            + 3.0 * MhyList[i] * (Ri1 - Ri_1) * RhoList[i] * Ri_1_2 * Ri_3_minus_Ri_1_3_inv
        )

        Hydromat[i, i] = (
            8.0 * Ri * (pList[i+1] - pList[i])
            + 20.0 * Ri_4 * (pList[i] * Ri_3_minus_Ri_1_3_inv + pList[i+1] * Ri1_3_minus_Ri_3_inv)
            + 3.0 * MhyList[i] * (Ri1 - Ri_1) * Ri_2 * (-RhoList[i] * Ri_3_minus_Ri_1_3_inv + RhoList[i+1] * Ri1_3_minus_Ri_3_inv)
        )

        Hydromat[i, i+1] = (
            MhyList[i] * (RhoList[i] + RhoList[i+1])
            - 20.0 * Ri_2 * pList[i+1] * Ri1_2 * Ri1_3_minus_Ri_3_inv
            - 3.0 * MhyList[i] * (Ri1 - Ri_1) * RhoList[i+1] * Ri1_2 * Ri1_3_minus_Ri_3_inv
        )

        Hydrob[i] = (
            -4.0 * Ri_2 * (pList[i+1] - pList[i])
            - MhyList[i] * (RhoList[i] + RhoList[i+1]) * (Ri1 - Ri_1)
        )

    last = NoLayers - 2
    Rl = RList[last]
    Rl_2 = Rl * Rl
    Rl_4 = Rl_2 * Rl_2
    Rl_1 = RList[last-1]
    Rl_1_2 = Rl_1 * Rl_1
    Rl1 = RList[last+1]
    Rl_Rl_1 = Rl - Rl_1
    Rl_Rl_1_sum_of_squares = Rl_2 + Rl * Rl_1 + Rl_1_2
    Rl_3_minus_Rl_1_3_inv = 1.0 / (Rl_Rl_1 * Rl_Rl_1_sum_of_squares)

    Hydromat[last, last-1] = (
        -MhyList[last] * (RhoList[last] + RhoList[last+1]) -
        20.0 * Rl_2 * pList[last] * Rl_1_2 * Rl_3_minus_Rl_1_3_inv +
        3.0 * MhyList[last] * (Rl1 - Rl_1) * RhoList[last] * Rl_1_2 * Rl_3_minus_Rl_1_3_inv
    )

    Hydromat[last, last] = (
        8.0 * Rl * (pList[last+1] - pList[last]) +
        20.0 * Rl_4 * pList[last] * Rl_3_minus_Rl_1_3_inv -
        3.0 * MhyList[last] * (Rl1 - Rl_1) * Rl_2 * RhoList[last] * Rl_3_minus_Rl_1_3_inv
    )

    Hydrob[last] = (
        -4.0 * Rl_2 * (pList[last+1] - pList[last]) -
        MhyList[last] * (RhoList[last] + RhoList[last+1]) * (Rl1 - Rl_1)
    )


@njit(fastmath=True)
def numba_calc_delta_rho_p(deltaRho, deltap, deltaR, RList, pList, RhoList, NoLayers):
    R0 = RList[0]
    R0_2 = R0 * R0
    R0_3 = R0_2 * R0
    deltaRho[0] = -3.0 * RhoList[0] * R0_2 * deltaR[0] / R0_3
    deltap[0] = -5.0 * pList[0] * R0_2 * deltaR[0] / R0_3

    for i in range(1, NoLayers - 1):
        Ri = RList[i]
        Ri_2 = Ri * Ri
        Ri_1 = RList[i-1]
        Ri_1_2 = Ri_1 * Ri_1
        Ri_Ri_1 = Ri - Ri_1
        Ri_3_minus_Ri_1_3_inv = 1.0 / (Ri_Ri_1 * (Ri_2 + Ri * Ri_1 + Ri_1_2))

        deltaRho[i] = -3.0 * RhoList[i] * (Ri_2 * deltaR[i] - Ri_1_2 * deltaR[i-1]) * Ri_3_minus_Ri_1_3_inv
        deltap[i] = -5.0 * pList[i] * (Ri_2 * deltaR[i] - Ri_1_2 * deltaR[i-1]) * Ri_3_minus_Ri_1_3_inv


@njit(fastmath=True)
def numba_update_profiles(RList, RhoList, MList, MhyList, uList, vList, aList, LList, mass_norm, scale_norm, a, b, c, sigma):
    NoLayers = len(RList)
    sigma_2 = sigma * sigma

    for i in range(NoLayers - 1):
        MhyList[i] = MList[i] + mass_norm * (1.0 + scale_norm**2 / RList[i]**2)**-1.5

    uList[:] = 1.5 * aList * (RhoList ** (2.0 / 3.0))
    vList[:] = np.sqrt(2.0 / 3.0) * np.sqrt(uList)

    R0 = RList[0]
    R0_2 = R0 * R0
    v0 = vList[0]
    v0_2 = v0 * v0
    v0_3 = v0_2 * v0
    v1 = vList[1]
    v1_2 = v1 * v1
    v1_3 = v1_2 * v1

    LList[0] = (
        -(uList[1] - uList[0]) / RList[1] * R0_2 * a * b * c * sigma *
        (RhoList[0] * v0_3 / (a * c * sigma_2 * RhoList[0] * v0_2 + b) +
         RhoList[1] * v1_3 / (a * c * sigma_2 * RhoList[1] * v1_2 + b))
    )

    for i in range(1, NoLayers - 1):
        Ri = RList[i]
        Ri_2 = Ri * Ri
        Ri1 = RList[i+1]
        Ri_1 = RList[i-1]

        vi = vList[i]
        vi_2 = vi * vi
        vi_3 = vi_2 * vi

        vi1 = vList[i+1]
        vi1_2 = vi1 * vi1
        vi1_3 = vi1_2 * vi1

        LList[i] = (
            -(uList[i+1] - uList[i]) / (Ri1 - Ri_1) * Ri_2 * a * b * c * sigma *
            (RhoList[i] * vi_3 / (a * c * sigma_2 * RhoList[i] * vi_2 + b) +
             RhoList[1 if False else i+1] * vi1_3 / (a * c * sigma_2 * RhoList[i+1] * vi1_2 + b))
        )

In [ ]:
class Simulator:
    def __init__(self, state):
        self.state = state

    def initialize(self, state):
        state.initialize()
        NoLayers = self.state.NoLayers

        self.deltaUcoeff = np.zeros(NoLayers)
        self.deltaU = np.zeros(NoLayers)
        self.hydrostaticI = np.zeros(NoLayers)
        self.hydrostaticM = np.zeros(NoLayers)
        self.hydrostaticF = np.zeros(NoLayers)

        self.Hydromat = np.zeros((NoLayers-1, NoLayers-1))
        self.Hydrob = np.zeros(NoLayers-1)
        self.deltaR = np.zeros(NoLayers-1)
        self.deltap = np.zeros(NoLayers-1)
        self.deltaRho = np.zeros(NoLayers-1)

        self.total_time = total_time

        self.file = open(output_file, "w")
        self.file.write(
            'Input profile name:' + str(tag) + '\n' +
            'Initial time:' + str(self.total_time) + '\n' +
            'Number of layers:' + str(state.NoLayers) + '\n' +
            'Initial r inner most:' + str(state.RList[0]) + '\n' +
            'Initial r outer most:' + str(state.RList[NoLayers-1]) + '\n' +
            'Initial rho inner most:' + str(state.RhoList[0]) + '\n' +
            'Initial rho outer most:' + str(state.RhoList[NoLayers-1]) + '\n' +
            'Total step:' + str(total_steps) + '\n' +
            'Save step:' + str(save_steps) + '\n' +
            'Abs(delta u/u):' + str(epsilon) + '\n' +
            'Cross section (sigma):' + str(sigma) + '\n' +
            'Conduction parameter (a,b,c):' + str(a) + ',' + str(b) + ',' + str(c) + '\n' +
            'Baryon parameter (mass_norm,scale_norm):' + str(mass_norm) + ',' + str(scale_norm) + '\n' +
            'time, step, SIDM radius, SIDM density, SIDM enclosed mass, SIDM internal energy, SIDM luminosity\n' +
            str(self.total_time) + ' ' + str(0) + '\n' +
            str(state.RList.T) + '\n' + str(state.RhoList.T) + '\n' + str(state.MList) + '\n' + str(state.uList) + '\n' + str(state.LList) + '\n'
        )
        print('Evolution with baryon starts!')

    def runSimulation(self, state):
        currentSaveStep = save_steps

        for tstep in range(1, total_steps + 1):
            try:
                deltat = self.performConductionStep(state)
                self.total_time = self.total_time + deltat

                self.performRelaxationStep(state)
                self.updateProfiles(state)

                if state.RhoList[0] >= 1e6 and currentSaveStep != 1:
                    currentSaveStep = 1
                    print('Density threshold 1e6 reached! Save frequency increased to every step.')

                if state.checkForAbnormalState():
                    break

                if (tstep % currentSaveStep) == 0:
                    self.saveResults(state, tstep)
                    print('Saved at time=', self.total_time, ',', 'step=', tstep)

            except Exception as e:
                print('Error during simulation step:', tstep, ':', e)
                traceback.print_exc()
                break

        self.file.close()
        print('Evolution ends')

    def performConductionStep(self, state):
        return numba_conduction_step(
            state.LList, state.MList, state.uList, 
            state.RhoList, state.pList, state.aList, epsilon
        )

    def performRelaxationStep(self, state):
        NoLayers = self.state.NoLayers
        for hstep in range(0, relaxation_steps):
            try:
                numba_setup_matrix(
                    self.Hydromat, self.Hydrob, 
                    state.RList, state.pList, state.MhyList, state.RhoList, NoLayers
                )
                self.deltaR = np.linalg.solve(self.Hydromat, self.Hydrob)
                
                numba_calc_delta_rho_p(
                    self.deltaRho, self.deltap, self.deltaR, 
                    state.RList, state.pList, state.RhoList, NoLayers
                )

                i = np.arange(0, NoLayers - 1)
                state.RhoList[i] += self.deltaRho[i]
                state.pList[i] += self.deltap[i]
                state.RList[i] += self.deltaR[i]

            except Exception as e:
                print('Error during relaxation step:', hstep, ':', e)
                traceback.print_exc()
                break

    def updateProfiles(self, state):
        numba_update_profiles(
            state.RList, state.RhoList, state.MList, state.MhyList, 
            state.uList, state.vList, state.aList, state.LList, 
            mass_norm, scale_norm, a, b, c, sigma
        )

    def saveResults(self, state, tstep):
        self.file.write(
            str(self.total_time) + ' ' + str(tstep) + '\n' +
            str(state.RList.T) + '\n' +
            str(state.RhoList.T) + '\n' +
            str(state.MList.T) + '\n' +
            str(state.uList.T) + '\n' +
            str(state.LList.T) + '\n'
        )

# Main

In [ ]:
def main():
  start=time.time()
  try:
    display()
    state=SimulationState()
    simulator=Simulator(state)
    simulator.initialize(state)
    simulator.runSimulation(state)
    end=time.time()
    print('Computation time:'+str(end-start)+'s')
  except Exception as e:
    print('Error:',e)

In [ ]:
main()

In [ ]:
import matplotlib.pyplot as plt
import re

# 1. File Path - Update this to match your actual output file path

def read_gravothermal_output(filename):
    with open(output_file, 'r') as f:
        lines = f.readlines()

    # Skip the 14 header lines
    raw_data = "".join(lines[14:])

    # Extract all arrays enclosed in brackets [ ... ]
    # re.DOTALL ensures it captures arrays that span across multiple newlines
    array_strings = re.findall(r'\[(.*?)\]', raw_data, re.DOTALL)

    # Remove all the arrays from the text to leave behind only the time/step lines
    time_step_text = re.sub(r'\[.*?\]', '', raw_data, flags=re.DOTALL)

    # Clean up empty lines to get a list of "time step" strings
    time_lines = [line.strip() for line in time_step_text.split('\n') if line.strip()]

    snapshots = []

    # Each time snapshot consists of 1 time line and 5 arrays (R, Rho, M, u, L)
    for i, t_line in enumerate(time_lines):
        # Parse time and step
        t_val, step_val = map(float, t_line.split())

        # Get the corresponding 5 arrays using the index multiplier
        idx = i * 5
        r_arr = np.fromstring(array_strings[idx], sep=' ')
        rho_arr = np.fromstring(array_strings[idx+1], sep=' ')
        m_arr = np.fromstring(array_strings[idx+2], sep=' ')
        u_arr = np.fromstring(array_strings[idx+3], sep=' ')
        l_arr = np.fromstring(array_strings[idx+4], sep=' ')

        snapshots.append({
            'time': t_val,
            'step': int(step_val),
            'r': r_arr,
            'rho': rho_arr,
            'M': m_arr,
            'u': u_arr,
            'L': l_arr
        })

    return snapshots

# 2. Extract Data
snapshots = read_gravothermal_output(output_file)
print(f"Successfully loaded {len(snapshots)} time snapshots.")

# 3. Create Plots
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

# Generate a color map so early times are one color and late times are another
colors = plt.cm.viridis(np.linspace(0, 1, len(snapshots)))

for i, snap in enumerate(snapshots):
    t_label = f"t = {snap['time']:.4f}" if snap['time'] > 0 else "t = 0.0 (Initial)"

    # Panel 1: Internal Energy (u) vs Radius (r)
    axes[0].plot(snap['r'], snap['u'], label=t_label, color=colors[i], lw=1.5)

    # Panel 2: Luminosity (L) vs Radius (r)
    axes[1].plot(snap['r'], snap['L'], label=t_label, color=colors[i], lw=1.5)

    axes[2].plot(snap['r'], snap['rho'], label=t_label, color=colors[i], lw=1.5, )

# 4. Format Panel 1 (Energy)
axes[0].set_title('Internal Energy ($u$) vs Radius', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Radius ($r$)', fontsize=12)
axes[0].set_ylabel('Internal Energy ($u$)', fontsize=12)
axes[0].set_xscale('log')
axes[0].grid(True, which="both", ls=":", alpha=0.6)

# 5. Format Panel 2 (Luminosity)
axes[1].set_title('Luminosity ($L$) vs Radius', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Radius ($r$)', fontsize=12)
axes[1].set_ylabel('Luminosity Flux ($L$)', fontsize=12)
axes[1].set_xscale('log')
axes[1].grid(True, which="both", ls=":", alpha=0.6)


axes[2].set_title('Density ($rho$) vs Radius', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Radius ($r$)', fontsize=12)
axes[2].set_ylabel('Density ($rho$)', fontsize=12)
axes[2].set_xscale('log')
axes[2].grid(True, which="both", ls=":", alpha=0.6)
axes[2].set_yscale('log')

#plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import re

# 1. Read the Output File
def read_gravothermal_output(filename):
    with open(filename, 'r') as f:
        lines = f.readlines()

    # Skip the 14 header lines
    raw_data = "".join(lines[14:])

    # Extract all arrays enclosed in brackets [ ... ]
    array_strings = re.findall(r'\[(.*?)\]', raw_data, re.DOTALL)

    # Remove all the arrays from the text to leave behind only the time/step lines
    time_step_text = re.sub(r'\[.*?\]', '', raw_data, flags=re.DOTALL)
    time_lines = [line.strip() for line in time_step_text.split('\n') if line.strip()]

    snapshots = []

    # Each time snapshot consists of 1 time line and 5 arrays (R, Rho, M, u, L)
    for i, t_line in enumerate(time_lines):
        t_val, step_val = map(float, t_line.split())

        idx = i * 5
        r_arr   = np.fromstring(array_strings[idx], sep=' ')
        rho_arr = np.fromstring(array_strings[idx+1], sep=' ')
        m_arr   = np.fromstring(array_strings[idx+2], sep=' ')
        u_arr   = np.fromstring(array_strings[idx+3], sep=' ')
        l_arr   = np.fromstring(array_strings[idx+4], sep=' ')

        snapshots.append({
            'time': t_val,
            'step': int(step_val),
            'r': r_arr,
            'rho': rho_arr,
            'M': m_arr,
            'u': u_arr,
            'L': l_arr
        })

    return snapshots

# 2. Extract Data
snapshots = read_gravothermal_output(output_file)
print(f"Successfully loaded {len(snapshots)} time snapshots.")

# 3. Create Plots (1 row, 2 columns)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Generate a color map so early times are one color and late times are another
colors = plt.cm.viridis(np.linspace(0, 1, len(snapshots)))

for i, snap in enumerate(snapshots):
    t_label = f"t = {snap['time']:.4f}" if snap['time'] > 0 else "t = 0.0 (Initial)"

    # Calculate Velocity Dispersion from Internal Energy: v = sqrt((2/3) * u)
    v_dispersion = 2.0 / 3.0 * snap['u']

    # Panel 1: Density (rho) vs Radius (r)
    axes[0].plot(snap['r'], snap['rho'], label=t_label, color=colors[i], lw=1.5)

    # Panel 2: Velocity Dispersion (v) vs Radius (r)
    axes[1].plot(snap['r'], v_dispersion, label=t_label, color=colors[i], lw=1.5)

# 4. Format Panel 1 (Density)
axes[0].set_title('Density Profile ($\\rho$) vs Radius', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Radius ($r$)', fontsize=12)
axes[0].set_ylabel('Density ($\\rho$)', fontsize=12)
axes[0].set_xscale('log')
axes[0].set_yscale('log')


axes[0].grid(True, which="both", ls=":", alpha=0.6)
#axes[0].legend(loc='best', fontsize=9)

# 5. Format Panel 2 (Velocity Dispersion)
axes[1].set_title('Velocity Dispersion ($v2$) vs Radius', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Radius ($r$)', fontsize=12)
axes[1].set_ylabel('Velocity Dispersion ($v2$)', fontsize=12)
axes[1].set_xscale('log')

axes[1].grid(True, which="both", ls=":", alpha=0.6)
#axes[1].legend(loc='best', fontsize=9)

plt.tight_layout()
plt.show()